# Task 7: Responsible and Sustainable AI

## 7.1 Responsible AI Objective

This task evaluates the traffic intelligence solution from the perspectives of bias, fairness, governance and sustainability.

No new machine-learning model is developed in this task. Instead, the analysis reviews the limitations and operational risks of the data, proxy labels, predictive models, recommendation system and deployment simulation developed in the earlier tasks.

The assessment focuses on:

- Sampling and coverage limitations in the traffic dataset.
- Risks associated with using proxy labels when real accident data are unavailable.
- The possibility that model errors are distributed unevenly across traffic, time and weather conditions.
- Human oversight and governance requirements before real-world deployment.
- Model versioning, monitoring and operational safeguards.
- Computational and environmental trade-offs associated with the modelling choices.

The purpose is to identify where the system may produce unreliable or unfair outcomes and to define safeguards that would be required before the solution could be trusted for real-world mobility decision-making.

## 7.2 Bias and Fairness Report

### 7.2.1 Sampling and Coverage Limitations

The dataset does not provide representative coverage of an entire city or road network. It contains approximately 48,000 hourly observations from a **single westbound I-94 traffic corridor near Minneapolis–St Paul**.

Consequently, patterns learned from this dataset should not automatically be generalised to:

- Other directions of travel on I-94.
- Other roads or intersections.
- Other neighbourhoods or cities.
- Different transport networks or populations.

The dataset also covers approximately **2012 to 2018**. Traffic behaviour may have changed since this period because of population changes, infrastructure development, remote-working patterns, public transport use and other changes in mobility behaviour.

Therefore, strong performance on this historical dataset does not demonstrate that the models would perform equally well on current or geographically different traffic conditions.

This limitation also applies to the recommendation system. Its recommendations describe historically lower-traffic periods on the observed corridor; they are not city-wide route recommendations or real-time traffic forecasts.

### 7.2.2 Proxy Labels and Their Risks

A real accident dataset was **not used in this project**.

Because actual accident outcomes were unavailable, the classification task used a documented proxy for accident risk. Congestion categories were derived from `traffic_volume`, and the high-risk proxy combined high or severe congestion with severe or low-visibility weather conditions.

This proxy enables a classification exercise, but it has important limitations.

High traffic volume or congestion is **not equivalent to an accident**. Similarly, severe weather combined with congestion does not prove that an accident occurred. The proxy therefore represents traffic conditions that may be associated with increased operational concern rather than observed accident likelihood.

This creates a risk of **label bias** because the definition of the target already embeds assumptions about which conditions are considered risky. A model trained on this proxy may learn to reproduce those assumptions rather than discover actual accident risk.

For this reason, classification outputs from the proxy model should not be presented as actual accident probabilities or used independently for safety-critical decisions.

The same distinction applies to the recommendation system. It recommends historically lower-traffic travel periods. A lower-traffic period should **not** be interpreted as a safer route or evidence of lower accident risk.

### 7.2.3 Uneven Distribution of Model Errors

Overall performance metrics such as MAE, R², accuracy or F1-score do not guarantee that prediction errors are distributed equally across all operating conditions.

Errors may be larger during:

- Weekday commuting and peak-traffic periods.
- Holidays and other relatively uncommon calendar conditions.
- Rare or severe weather conditions.
- Low-visibility conditions such as fog.
- Very uncommon weather categories such as squall.
- Periods whose traffic behaviour differs from the historical training data.

The recommendation analysis already demonstrated substantial differences in historical coverage between weather categories. Common conditions such as Clear, Clouds and Rain contain many observations, while rare conditions contain substantially less evidence. Predictions and recommendations for sparsely represented conditions should therefore be treated with greater caution.

The project also uses a **random train-test split** for the regression experiment. Although this provides a reproducible model comparison, observations from different historical periods can appear in both the training and test sets. As a result, test performance may appear stronger than performance on a genuinely future, unseen time period.

Before real-world deployment, the model should therefore be evaluated using time-based validation and subgroup performance analysis across day type, hour, holidays, weather conditions and other operationally important groups.

## 7.3 Governance and Human Oversight

A mobility AI system should not be trusted for real-world operational decisions solely because it performs well on historical test data.

Before deployment, appropriate human and organisational oversight should be established.

### Human Review and Approval

A responsible deployment process should require an authorised traffic or mobility specialist to review the model before it is used for operational decision-making.

The reviewer should confirm:

- That the model has been evaluated on sufficiently recent and representative data.
- That performance has been examined across important subgroups and operating conditions.
- That proxy outputs are not presented as observed accident probabilities.
- That the intended use of the model remains within the scope supported by the data.
- That limitations are clearly communicated to users and decision-makers.

High-impact or safety-related decisions should not be made automatically from the model output without appropriate human review.

### Model Versioning and Auditability

The MLflow workflow developed in Tasks 4 and 6 provides a foundation for model governance.

Each deployed model should have a clearly identified version together with its:

- Training configuration.
- Evaluation metrics.
- Input feature structure.
- Deployment status.
- Monitoring history.

The Random Forest deployment candidate is identified as **v2**, with the Task 4 MAE of **271.3004** and R² of **0.9442** retained as the authoritative historical performance baseline.

Maintaining this information makes it possible to determine which model produced a prediction and to compare future versions against an approved reference.

### Monitoring, Alerts and Operational Response

Deployment should include continuous monitoring rather than assuming that historical performance will remain stable.

Task 6 demonstrated a monitoring rule based on prediction error. The Task 4 MAE of **271.3004** was used as the reference baseline, with a simulated alert threshold of **1.5 times the baseline MAE**.

When monitored performance remains within the threshold, the system reports:

**PASS / Normal**

When the threshold is exceeded, the system reports:

**ALERT / Requires investigation**

In a real deployment, an ALERT should trigger human investigation. Depending on the severity and persistence of the problem, automated recommendations or predictions should be suspended, restricted or reverted to a previously approved model until the cause has been understood.

Monitoring thresholds should also be reviewed periodically rather than treated as permanent values.

### Change Control

Updates to data preparation, features, model parameters, proxy-label definitions or recommendation logic should be treated as controlled changes.

A revised model should be tested and approved before replacing the currently accepted version. Previous model versions, evaluation results and monitoring records should remain available for audit and rollback.

## 7.4 Sustainability and Resource Trade-offs

Responsible AI also requires consideration of the computational resources and environmental cost associated with model development and operation.

### Model Complexity and Performance

The project evaluated models with different levels of complexity.

For traffic-volume regression, the Random Forest model achieved an MAE of approximately **271.30** and an R² of approximately **0.9442**, providing strong predictive performance while remaining practical to train and deploy on conventional computing hardware.

The project also explored deep learning in an earlier task. Deep neural models can provide value for complex or sequential problems, but they generally require greater computational resources for training, tuning and repeated experimentation.

A more complex model should therefore not be selected simply because it is more sophisticated. Additional computational cost should be justified by a meaningful improvement in predictive or operational value.

### Computational and Environmental Trade-offs

Repeated training of large or deep models increases:

- CPU or GPU usage.
- Memory requirements.
- Training time.
- Electricity consumption.
- Storage requirements for experiments and model artifacts.

For a traffic system that may require periodic retraining, these costs can accumulate over time.

The Random Forest deployment candidate provides a reasonable balance between predictive performance and computational requirements for this project. Where a simpler model provides adequate performance, unnecessary use of substantially larger models should be avoided.

### Sustainable MLOps Practices

A more sustainable operational approach would include:

- Retraining models when monitoring indicates a meaningful need rather than on an unnecessarily frequent schedule.
- Reusing validated data pipelines and engineered features.
- Tracking experiments so that unsuccessful configurations are not repeatedly tested without justification.
- Retaining only model artifacts and logs required for reproducibility, governance and rollback.
- Comparing the performance benefit of more computationally expensive models with their additional resource cost.

Sustainability should therefore be considered together with accuracy, reliability and operational value when selecting and maintaining mobility AI models.


## 7.5 Responsible AI Recommendations

Based on the limitations identified above, the following safeguards are recommended before this traffic intelligence solution is considered for real-world use.

### Data and Fairness

- Collect more recent traffic data before operational deployment.
- Expand coverage beyond a single westbound I-94 corridor to improve geographic representativeness.
- Obtain real accident records before interpreting any model output as accident likelihood.
- Continue to describe the current accident-risk target explicitly as a **proxy label**.
- Evaluate model performance separately across weekday and weekend periods, peak and off-peak hours, holidays and weather conditions.
- Apply additional caution to rare conditions such as fog, squall and other sparsely represented weather categories.
- Use time-based validation to assess performance on genuinely future observations rather than relying only on a random train-test split.

### Governance and Oversight

- Require human review and approval before a model is promoted for operational use.
- Maintain model versions, parameters, metrics and deployment artifacts through MLflow or an equivalent model-governance system.
- Monitor deployed performance and investigate meaningful deterioration.
- Suspend or restrict automated predictions and recommendations when monitoring produces an **ALERT / Requires investigation** status.
- Preserve previous approved model versions so that the system can be rolled back if necessary.
- Document changes to data preparation, proxy labels, features and recommendation logic.

### Sustainability

- Prefer the least computationally expensive model that provides adequate operational performance.
- Avoid unnecessary retraining when monitoring indicates that the existing model remains reliable.
- Reuse validated data pipelines and engineered features rather than repeatedly recreating equivalent processing.
- Consider the additional energy, hardware, training time and storage requirements before adopting substantially more complex models.

These safeguards would improve the reliability, accountability and sustainability of the system while reducing the risk of applying historical model outputs beyond the conditions supported by the available data.


## 7.6 Final Assessment

This responsible AI assessment identifies important limitations that must be considered alongside the technical results of the project.

The dataset has limited geographic and temporal coverage, and the available historical observations should not be assumed to represent current or city-wide traffic behaviour.

No real accident dataset was used. The accident-risk classification task relies on a documented proxy derived from congestion and weather conditions. Consequently, the resulting classification outputs must not be interpreted as observed accident probabilities.

Model errors may also be distributed unevenly across time periods, holidays and weather conditions, particularly where historical observations are sparse. Future deployment should therefore include subgroup evaluation and time-based validation.

Before the system is trusted for real-world mobility decisions, it should be subject to human review, controlled model versioning, ongoing monitoring and an operational response to alerts. Model outputs should support rather than replace accountable human decision-making.

Finally, model complexity should be justified by operational value. More computationally demanding models increase training, hardware and energy requirements, so predictive performance should be considered together with reliability, governance and sustainability.

Overall, the project demonstrates a responsible AI approach by explicitly documenting data limitations, proxy-label risks, potential uneven error distribution, governance requirements and computational trade-offs rather than treating predictive performance alone as sufficient evidence for deployment.